<a href="https://colab.research.google.com/github/sswess122-star/full-outer-join/blob/main/G6_DV_L13_Window_Functions_SQLite3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEMNest Academy | Data Science Track | Grade 6 | Dataville
## G6-DV-L13 — Window Functions
### Unit 2 — SQL Intermediate | Colab + SQLite3 + Python + pandas

**Teacher lesson note**

**Main idea:** A window function calculates something about related rows without removing the individual rows.

Today:
- `ROW_NUMBER()` — number rows
- `RANK()` — rank rows
- `DENSE_RANK()` — rank without gaps
- `LAG()` — look backwards
- `LEAD()` — look forwards

**Grade 6 rule:** Focus on `OVER`, `PARTITION BY`, and `ORDER BY`. Do not introduce window frames yet.

## Learning Objectives

Students should be able to:

1. Use `ROW_NUMBER()` to number students by score.
2. Use `RANK() OVER (PARTITION BY ...)` to rank students within a track.
3. Explain `RANK()` versus `DENSE_RANK()`.
4. Use `LAG()` and `LEAD()` with assessment data.
5. Combine a window function with a CTE to solve a top-N-per-group problem.
6. Connect SQL window functions to pandas `rank()` and `shift()`.

## Dr Osei's Data Brief — Tutor Reads Aloud

> “Imagine that we want to rank every student in Dataville. We want the ranking, but we also want every student's original information to remain visible.
>
> That is where window functions help us. A window function looks at related rows, calculates something, and adds the answer to each row. It does not collapse the rows like `GROUP BY`.
>
> Today we will use `ROW_NUMBER()` to number students, `RANK()` to rank them, and `LAG()` to compare an assessment with the previous assessment.
>
> You will also see the pandas connection: `rank()` and `shift()` solve similar problems after data reaches Python.”

## Why SQLite3 in Google Colab?

This Grade 6 version uses SQLite3 instead of PostgreSQL so students are not distracted by server setup.

- No PostgreSQL server is required.
- No username or password is required.
- Python's built-in `sqlite3` library connects directly.
- pandas can read SQL with `pd.read_sql_query()`.
- Modern SQLite supports the window functions used in this lesson.

**Important:** The database is created inside the Colab runtime. If the runtime restarts, run the setup cells again.

## Grade 5 Bridge

| Grade 5 / pandas | Grade 6 / SQL |
|---|---|
| `df['score'].rank()` | `RANK() OVER (...)` |
| `df.groupby('track')['score'].rank()` | `RANK() OVER (PARTITION BY track ...)` |
| `df['score'].shift(1)` | `LAG(score) OVER (...)` |
| `df['score'].shift(-1)` | `LEAD(score) OVER (...)` |

**Big idea:** `GROUP BY` usually reduces rows. Window functions normally keep every row and add a calculated column.

In [ ]:
import sqlite3
import pandas as pd

print("pandas:", pd.__version__)
print("SQLite:", sqlite3.sqlite_version)

conn = sqlite3.connect(":memory:")
print("SQLite database connected.")

pandas: 2.2.3
SQLite: 3.45.1
SQLite database connected.


# TASK 0 — Build the STEMNest Practice Database

We will use:

- `students` — student scores
- `enrolments` — connects students to courses
- `assessments` — scores over time

This is deliberately small so Grade 6 students can understand the data.

In [ ]:
conn.executescript('''
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    track TEXT NOT NULL,
    math_score INTEGER NOT NULL,
    science_score INTEGER NOT NULL,
    project_score INTEGER NOT NULL
);

CREATE TABLE enrolments (
    enrolment_id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    course_name TEXT NOT NULL,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
);

CREATE TABLE assessments (
    assessment_id INTEGER PRIMARY KEY,
    enrolment_id INTEGER NOT NULL,
    assessment_date TEXT NOT NULL,
    score REAL NOT NULL,
    FOREIGN KEY (enrolment_id) REFERENCES enrolments(enrolment_id)
);
''')

students = [
    (1, "Ada", "Dataville", 92, 88, 90),
    (2, "Ben", "Dataville", 85, 82, 86),
    (3, "Chika", "Dataville", 85, 79, 88),
    (4, "David", "Dataville", 72, 75, 78),
    (5, "Efe", "Mindspire", 95, 91, 94),
    (6, "Favour", "Mindspire", 88, 84, 89),
    (7, "Grace", "Mindspire", 88, 86, 87),
    (8, "Hassan", "Mindspire", 70, 73, 76),
    (9, "Ivy", "Webweave", 91, 89, 93),
    (10, "John", "Webweave", 76, 80, 82)
]

conn.executemany(
    '''INSERT INTO students
       (student_id, name, track, math_score, science_score, project_score)
       VALUES (?, ?, ?, ?, ?, ?)''',
    students
)

enrolments = [
    (1,1,"Python Foundations"), (2,2,"Python Foundations"),
    (3,3,"Python Foundations"), (4,4,"Python Foundations"),
    (5,5,"Data Analytics"), (6,6,"Data Analytics"),
    (7,7,"Data Analytics"), (8,8,"Data Analytics"),
    (9,9,"Web Development"), (10,10,"Web Development")
]

conn.executemany(
    '''INSERT INTO enrolments
       (enrolment_id, student_id, course_name)
       VALUES (?, ?, ?)''',
    enrolments
)

assessments = [
    (1,1,"2026-01-10",68),(2,1,"2026-02-10",75),(3,1,"2026-03-10",84),
    (4,2,"2026-01-10",72),(5,2,"2026-02-10",70),(6,2,"2026-03-10",78),
    (7,3,"2026-01-10",80),(8,3,"2026-02-10",84),(9,3,"2026-03-10",86),
    (10,4,"2026-01-10",60),(11,4,"2026-02-10",64),(12,4,"2026-03-10",62),
    (13,5,"2026-01-10",78),(14,5,"2026-02-10",86),(15,5,"2026-03-10",94),
    (16,6,"2026-01-10",70),(17,6,"2026-02-10",76),(18,6,"2026-03-10",81),
    (19,7,"2026-01-10",82),(20,7,"2026-02-10",80),(21,7,"2026-03-10",88),
    (22,8,"2026-01-10",65),(23,8,"2026-02-10",68),(24,8,"2026-03-10",66),
    (25,9,"2026-01-10",75),(26,9,"2026-02-10",85),(27,9,"2026-03-10",91),
    (28,10,"2026-01-10",69),(29,10,"2026-02-10",74),(30,10,"2026-03-10",79)
]

conn.executemany(
    '''INSERT INTO assessments
       (assessment_id, enrolment_id, assessment_date, score)
       VALUES (?, ?, ?, ?)''',
    assessments
)

conn.commit()

students_df = pd.read_sql_query("SELECT * FROM students", conn)
enrolments_df = pd.read_sql_query("SELECT * FROM enrolments", conn)
assessments_df = pd.read_sql_query("SELECT * FROM assessments", conn)

display(students_df)

,student_id,name,track,math_score,science_score,project_score
0,1,Ada,Dataville,92,88,90
1,2,Ben,Dataville,85,82,86
2,3,Chika,Dataville,85,79,88
3,4,David,Dataville,72,75,78
4,5,Efe,Mindspire,95,91,94
5,6,Favour,Mindspire,88,84,89
6,7,Grace,Mindspire,88,86,87
7,8,Hassan,Mindspire,70,73,76
8,9,Ivy,Webweave,91,89,93
9,10,John,Webweave,76,80,82


In [ ]:
enrolments_df = pd.read_sql_query("SELECT * FROM enrolments", conn)
display(enrolments_df)

,enrolment_id,student_id,course_name
0,1,1,Python Foundations
1,2,2,Python Foundations
2,3,3,Python Foundations
3,4,4,Python Foundations
4,5,5,Data Analytics
5,6,6,Data Analytics
6,7,7,Data Analytics
7,8,8,Data Analytics
8,9,9,Web Development
9,10,10,Web Development


In [ ]:
assessments_df = pd.read_sql_query("SELECT * FROM assessments", conn)
display(assessments_df)

,assessment_id,enrolment_id,assessment_date,score
0,1,1,2026-01-10,68.0
1,2,1,2026-02-10,75.0
2,3,1,2026-03-10,84.0
3,4,2,2026-01-10,72.0
4,5,2,2026-02-10,70.0
5,6,2,2026-03-10,78.0
6,7,3,2026-01-10,80.0
7,8,3,2026-02-10,84.0
8,9,3,2026-03-10,86.0
9,10,4,2026-01-10,60.0


# TASK 1 — `ROW_NUMBER()`

Question:

> Number all students from the highest math score to the lowest.

Pattern:

```sql
ROW_NUMBER() OVER (ORDER BY math_score DESC)
```

**Teacher point:** Every student remains in the result. We only add a new number.

In [ ]:
sql = '''
SELECT
    name,
    track,
    math_score,
    ROW_NUMBER() OVER (
        ORDER BY math_score DESC, name
    ) AS global_number
FROM students
ORDER BY global_number;
'''

df_row_number = pd.read_sql_query(sql, conn)
display(df_row_number)

,name,track,math_score,global_number
0,Efe,Mindspire,95,1
1,Ada,Dataville,92,2
2,Ivy,Webweave,91,3
3,Favour,Mindspire,88,4
4,Grace,Mindspire,88,5
5,Ben,Dataville,85,6
6,Chika,Dataville,85,7
7,John,Webweave,76,8
8,David,Dataville,72,9
9,Hassan,Mindspire,70,10


# TASK 2 — `RANK()` WITHIN EACH TRACK

Question:

> Who is 1st, 2nd, 3rd, etc. inside each track?

- `PARTITION BY track` creates a separate ranking group for each track.
- `ORDER BY math_score DESC` puts the highest score first.
- `RANK()` assigns the rank.

In [ ]:
sql = '''
SELECT
    name,
    track,
    math_score,
    RANK() OVER (
        PARTITION BY track
        ORDER BY math_score DESC
    ) AS track_rank
FROM students
ORDER BY track, track_rank, name;
'''

df_track_rank = pd.read_sql_query(sql, conn)
display(df_track_rank)

,name,track,math_score,track_rank
0,Ada,Dataville,92,1
1,Ben,Dataville,85,2
2,Chika,Dataville,85,2
3,David,Dataville,72,4
4,Efe,Mindspire,95,1
5,Favour,Mindspire,88,2
6,Grace,Mindspire,88,2
7,Hassan,Mindspire,70,4
8,Ivy,Webweave,91,1
9,John,Webweave,76,2


## `RANK()` versus `DENSE_RANK()`

For scores:

**95, 90, 90, 80, 75**

`RANK()` → **1, 2, 2, 4, 5**

`DENSE_RANK()` → **1, 2, 2, 3, 4**

**Teacher explanation:** `RANK()` leaves a gap after a tie. `DENSE_RANK()` does not.

In [ ]:
sql = '''
SELECT
    name,
    track,
    math_score,
    RANK() OVER (
        PARTITION BY track
        ORDER BY math_score DESC
    ) AS rank_value,
    DENSE_RANK() OVER (
        PARTITION BY track
        ORDER BY math_score DESC
    ) AS dense_rank_value
FROM students
ORDER BY track, math_score DESC, name;
'''

df_rank_compare = pd.read_sql_query(sql, conn)
display(df_rank_compare)

,name,track,math_score,rank_value,dense_rank_value
0,Ada,Dataville,92,1,1
1,Ben,Dataville,85,2,2
2,Chika,Dataville,85,2,2
3,David,Dataville,72,4,3
4,Efe,Mindspire,95,1,1
5,Favour,Mindspire,88,2,2
6,Grace,Mindspire,88,2,2
7,Hassan,Mindspire,70,4,3
8,Ivy,Webweave,91,1,1
9,John,Webweave,76,2,2


## pandas Bridge — `rank()`

The SQL idea can also be expressed in pandas:

```python
df.groupby('track')['math_score'].rank(
    ascending=False,
    method='min'
)
```

SQL does the calculation while retrieving the data; pandas does it after the data is loaded.

In [ ]:
pandas_rank = students_df.copy()

pandas_rank["track_rank"] = (
    pandas_rank.groupby("track")["math_score"]
    .rank(ascending=False, method="min")
)

display(
    pandas_rank.sort_values(
        ["track", "track_rank", "name"]
    )[["name", "track", "math_score", "track_rank"]]
)

,name,track,math_score,track_rank
0,Ada,Dataville,92,1.0
1,Ben,Dataville,85,2.0
2,Chika,Dataville,85,2.0
3,David,Dataville,72,4.0
4,Efe,Mindspire,95,1.0
5,Favour,Mindspire,88,2.0
6,Grace,Mindspire,88,2.0
7,Hassan,Mindspire,70,4.0
8,Ivy,Webweave,91,1.0
9,John,Webweave,76,2.0


# TASK 3 — `LAG()`: LOOK AT THE PREVIOUS SCORE

Question:

> For every assessment, what was the student's previous assessment score?

`LAG(score)` looks backwards.

```sql
LAG(score) OVER (
    PARTITION BY student
    ORDER BY assessment_date
)
```

For the first assessment of a student there is no previous row, so SQL returns `NULL`.

In [ ]:
sql = '''
SELECT
    s.name,
    a.assessment_date,
    a.score,
    LAG(a.score) OVER (
        PARTITION BY s.student_id
        ORDER BY a.assessment_date
    ) AS previous_score
FROM assessments AS a
JOIN enrolments AS e
    ON a.enrolment_id = e.enrolment_id
JOIN students AS s
    ON e.student_id = s.student_id
ORDER BY s.name, a.assessment_date;
'''

df_lag = pd.read_sql_query(sql, conn)
display(df_lag)

,name,assessment_date,score,previous_score
0,Ada,2026-01-10,68.0,NaN
1,Ada,2026-02-10,75.0,68.0
2,Ada,2026-03-10,84.0,75.0
3,Ben,2026-01-10,72.0,NaN
4,Ben,2026-02-10,70.0,72.0
5,Ben,2026-03-10,78.0,70.0
6,Chika,2026-01-10,80.0,NaN
7,Chika,2026-02-10,84.0,80.0
8,Chika,2026-03-10,86.0,84.0
9,David,2026-01-10,60.0,NaN


## Calculate Score Change

Now calculate:

**current score − previous score**

A positive number means the score increased. A negative number means it decreased.

In [ ]:
sql = '''
SELECT
    s.name,
    a.assessment_date,
    a.score,
    LAG(a.score) OVER (
        PARTITION BY s.student_id
        ORDER BY a.assessment_date
    ) AS previous_score,
    a.score - LAG(a.score) OVER (
        PARTITION BY s.student_id
        ORDER BY a.assessment_date
    ) AS score_change
FROM assessments AS a
JOIN enrolments AS e
    ON a.enrolment_id = e.enrolment_id
JOIN students AS s
    ON e.student_id = s.student_id
ORDER BY s.name, a.assessment_date;
'''

df_score_change = pd.read_sql_query(sql, conn)
display(df_score_change)

,name,assessment_date,score,previous_score,score_change
0,Ada,2026-01-10,68.0,NaN,NaN
1,Ada,2026-02-10,75.0,68.0,7.0
2,Ada,2026-03-10,84.0,75.0,9.0
3,Ben,2026-01-10,72.0,NaN,NaN
4,Ben,2026-02-10,70.0,72.0,-2.0
5,Ben,2026-03-10,78.0,70.0,8.0
6,Chika,2026-01-10,80.0,NaN,NaN
7,Chika,2026-02-10,84.0,80.0,4.0
8,Chika,2026-03-10,86.0,84.0,2.0
9,David,2026-01-10,60.0,NaN,NaN


# TASK 4 — `LEAD()`: LOOK AT THE NEXT SCORE

`LEAD()` is the opposite of `LAG()`.

- `LAG()` → previous row
- `LEAD()` → next row

Question:

> For every assessment, what is the student's next assessment score?

In [ ]:
sql = '''
SELECT
    s.name,
    a.assessment_date,
    a.score,
    LEAD(a.score) OVER (
        PARTITION BY s.student_id
        ORDER BY a.assessment_date
    ) AS next_score
FROM assessments AS a
JOIN enrolments AS e
    ON a.enrolment_id = e.enrolment_id
JOIN students AS s
    ON e.student_id = s.student_id
ORDER BY s.name, a.assessment_date;
'''

df_lead = pd.read_sql_query(sql, conn)
display(df_lead)

,name,assessment_date,score,next_score
0,Ada,2026-01-10,68.0,75.0
1,Ada,2026-02-10,75.0,84.0
2,Ada,2026-03-10,84.0,NaN
3,Ben,2026-01-10,72.0,70.0
4,Ben,2026-02-10,70.0,78.0
5,Ben,2026-03-10,78.0,NaN
6,Chika,2026-01-10,80.0,84.0
7,Chika,2026-02-10,84.0,86.0
8,Chika,2026-03-10,86.0,NaN
9,David,2026-01-10,60.0,64.0


## pandas Bridge — `shift()`

- SQL `LAG()` ↔ pandas `shift(1)`
- SQL `LEAD()` ↔ pandas `shift(-1)`

In [ ]:
pandas_assess = assessments_df.merge(
    enrolments_df[["enrolment_id", "student_id"]],
    on="enrolment_id",
    how="left"
).sort_values(["student_id", "assessment_date"])

pandas_assess["previous_score"] = (
    pandas_assess.groupby("student_id")["score"].shift(1)
)

pandas_assess["next_score"] = (
    pandas_assess.groupby("student_id")["score"].shift(-1)
)

display(
    pandas_assess[
        ["student_id", "assessment_date", "score",
         "previous_score", "next_score"]
    ]
)

,student_id,assessment_date,score,previous_score,next_score
0,1,2026-01-10,68.0,NaN,75.0
1,1,2026-02-10,75.0,68.0,84.0
2,1,2026-03-10,84.0,75.0,NaN
3,2,2026-01-10,72.0,NaN,70.0
4,2,2026-02-10,70.0,72.0,78.0
5,2,2026-03-10,78.0,70.0,NaN
6,3,2026-01-10,80.0,NaN,84.0
7,3,2026-02-10,84.0,80.0,86.0
8,3,2026-03-10,86.0,84.0,NaN
9,4,2026-01-10,60.0,NaN,64.0


# TASK 5 — TOP 2 STUDENTS PER TRACK

Question:

> Who are the top 2 students in every track?

`LIMIT 2` would return only two students overall.

Instead:

1. Rank students inside each track.
2. Store the ranked result in a CTE.
3. Keep ranks `<= 2`.

This is the **top-N-per-group** pattern.

In [ ]:
sql = '''
WITH ranked_students AS (
    SELECT
        name,
        track,
        math_score,
        RANK() OVER (
            PARTITION BY track
            ORDER BY math_score DESC
        ) AS track_rank
    FROM students
)
SELECT
    name,
    track,
    math_score,
    track_rank
FROM ranked_students
WHERE track_rank <= 2
ORDER BY track, track_rank, name;
'''

df_top2 = pd.read_sql_query(sql, conn)
display(df_top2)

,name,track,math_score,track_rank
0,Ada,Dataville,92,1
1,Ben,Dataville,85,2
2,Chika,Dataville,85,2
3,Efe,Mindspire,95,1
4,Favour,Mindspire,88,2
5,Grace,Mindspire,88,2
6,Ivy,Webweave,91,1
7,John,Webweave,76,2


# TASK 6 — TOP 3 BY TOTAL SCORE

Total score:

**math + science + project**

Then rank students within each track by total score.

In [ ]:
sql = '''
WITH scored_students AS (
    SELECT
        name,
        track,
        math_score + science_score + project_score AS total_score
    FROM students
),
ranked_students AS (
    SELECT
        name,
        track,
        total_score,
        RANK() OVER (
            PARTITION BY track
            ORDER BY total_score DESC
        ) AS track_rank
    FROM scored_students
)
SELECT
    name,
    track,
    total_score,
    track_rank
FROM ranked_students
WHERE track_rank <= 3
ORDER BY track, track_rank, name;
'''

df_top3_total = pd.read_sql_query(sql, conn)
display(df_top3_total)

,name,track,total_score,track_rank
0,Ada,Dataville,270,1
1,Ben,Dataville,253,2
2,Chika,Dataville,252,3
3,Efe,Mindspire,280,1
4,Favour,Mindspire,261,2
5,Grace,Mindspire,261,2
6,Ivy,Webweave,273,1
7,John,Webweave,238,2


# Student Practice — Guided Challenges

Try these before looking at the teacher solutions.

### Challenge 1
Number all students from highest science score to lowest.

**Hint:** `ROW_NUMBER()`

### Challenge 2
Rank students within each track by project score.

**Hint:** `RANK()` + `PARTITION BY`

### Challenge 3
For every assessment, show the previous score.

**Hint:** `LAG()`

### Challenge 4
For every assessment, show the next score.

**Hint:** `LEAD()`

### Challenge 5
Find the top 2 students in each track by total score.

**Hint:** CTE + `RANK()`

# Teacher Solutions to Guided Challenges

In [ ]:
# Challenge 1
q1 = '''
SELECT
    name,
    science_score,
    ROW_NUMBER() OVER (
        ORDER BY science_score DESC, name
    ) AS science_number
FROM students
ORDER BY science_number;
'''
display(pd.read_sql_query(q1, conn))

In [ ]:
# Challenge 2
q2 = '''
SELECT
    name,
    track,
    project_score,
    RANK() OVER (
        PARTITION BY track
        ORDER BY project_score DESC
    ) AS project_rank
FROM students
ORDER BY track, project_rank, name;
'''
display(pd.read_sql_query(q2, conn))

In [ ]:
# Challenge 3
q3 = '''
SELECT
    s.name,
    a.assessment_date,
    a.score,
    LAG(a.score) OVER (
        PARTITION BY s.student_id
        ORDER BY a.assessment_date
    ) AS previous_score
FROM assessments AS a
JOIN enrolments AS e
    ON a.enrolment_id = e.enrolment_id
JOIN students AS s
    ON e.student_id = s.student_id
ORDER BY s.name, a.assessment_date;
'''
display(pd.read_sql_query(q3, conn))

In [ ]:
# Challenge 4
q4 = '''
SELECT
    s.name,
    a.assessment_date,
    a.score,
    LEAD(a.score) OVER (
        PARTITION BY s.student_id
        ORDER BY a.assessment_date
    ) AS next_score
FROM assessments AS a
JOIN enrolments AS e
    ON a.enrolment_id = e.enrolment_id
JOIN students AS s
    ON e.student_id = s.student_id
ORDER BY s.name, a.assessment_date;
'''
display(pd.read_sql_query(q4, conn))

In [ ]:
# Challenge 5
q5 = '''
WITH totals AS (
    SELECT
        name,
        track,
        math_score + science_score + project_score AS total_score
    FROM students
),
ranked AS (
    SELECT
        name,
        track,
        total_score,
        RANK() OVER (
            PARTITION BY track
            ORDER BY total_score DESC
        ) AS track_rank
    FROM totals
)
SELECT
    name,
    track,
    total_score,
    track_rank
FROM ranked
WHERE track_rank <= 2
ORDER BY track, track_rank, name;
'''
display(pd.read_sql_query(q5, conn))

# Homework

## HW 1 — Notebook Task

Write two window-function queries:

1. For each track, find the student ranked **3rd by total score**.
2. Find students whose **most recent assessment score is lower than their previous assessment**, using `LAG()`.

For each:
- write the business question in Markdown;
- write the SQL;
- load the result into a pandas DataFrame;
- write one sentence explaining the result.

## HW 2 — Written Interpretation

Explain `RANK()` versus `DENSE_RANK()` using:

**95, 90, 90, 80, 75**

Show both outputs and explain the difference.

# Dr Osei's Closing Debrief — 7 minutes

> “Today you learned that window functions calculate across related rows without destroying the individual rows.
>
> `ROW_NUMBER()` numbers rows.
>
> `RANK()` ranks rows and leaves gaps after ties.
>
> `DENSE_RANK()` ranks rows without gaps.
>
> `LAG()` looks backwards.
>
> `LEAD()` looks forwards.
>
> The top-N-per-group pattern combines a CTE with a window function.”

### Real-world connection

Window functions are used in:
- student performance reports;
- sales leaderboards;
- customer activity analysis;
- financial reports;
- sports rankings;
- most-recent-record analysis;
- top-N reports.

### NEXT

**L14 — INSERT, UPDATE, DELETE and transaction basics.**

# Teacher Checklist

- [ ] Students understand that window functions do not collapse rows.
- [ ] Students understand `OVER`.
- [ ] Students can use `PARTITION BY`.
- [ ] Students can write `ROW_NUMBER()`.
- [ ] Students can write `RANK()`.
- [ ] Students can distinguish `RANK()` and `DENSE_RANK()`.
- [ ] Students can use `LAG()` and `LEAD()`.
- [ ] Students can explain the pandas `rank()` / `shift()` bridge.
- [ ] Students can use a CTE for top-N-per-group.
- [ ] Students complete the homework independently.

# End-of-Session Commit

**Folder:** `unit2-sql-intermediate/`

**Notebook:** `G6-DV-L13_Window_Functions_SQLite3.ipynb`

**Commit message:**

`L13 Window Functions: ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD, top-N-per-group`

**Colab note:** The SQLite database is in memory. If the runtime restarts, run the setup cells again.

In [ ]:
# Optional cleanup after the lesson
# conn.close()
# print("Database connection closed.")